In [1]:
import torch
import numpy as np
import time

def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using Apple Silicon GPU (MPS)


In [2]:
from helper import load_model_weights
from pathlib import Path
from Qwen3 import Qwen3Model, QWEN_CONFIG_06_B, QWEN_CONFIG_4_B_INSTRUCT

model_name = "Qwen/Qwen3-0.6B"
model_config = QWEN_CONFIG_06_B
tokenizer_path = Path("qwen3") / "qwen3-0.6B-base.json"
model_path = Path("qwen3") / "qwen3-0.6B-base.pth"


/Users/yuchenj/Personal/ml_practice/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer, model = load_model_weights(None, None, model_config, model_name, 'qwen3/')

Loading tokenizer from None...
Tokenizer Path does not exist, loading using model_name Qwen/Qwen3-0.6B
Saving tokenizer to qwen3/...
Tokenizer saved to qwen3/Qwen/Qwen3-0.6B.json
Using {'vocab_size': 151936, 'context_length': 40960, 'emb_dim': 1024, 'n_heads': 16, 'n_layers': 28, 'hidden_dim': 3072, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 1000000.0, 'dtype': torch.bfloat16} load model...
Successfully loaded model from config {'vocab_size': 151936, 'context_length': 40960, 'emb_dim': 1024, 'n_heads': 16, 'n_layers': 28, 'hidden_dim': 3072, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 1000000.0, 'dtype': torch.bfloat16}.
Loading weights from None...
Model does not exist, loading using model_name Qwen/Qwen3-0.6B


Loading weights: 100%|██| 311/311 [00:00<00:00, 1290.56it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Successfully loaded model from Qwen/Qwen3-0.6B
Saving model to qwen3/...
Model successfully saved to qwen3/Qwen/Qwen3-0.6B.pth


In [4]:
model = model.to(device)

In [5]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")
ds_split = ds['train'].train_test_split(train_size=0.8)

In [14]:
def prep_data_train(tokenizer, data):
    tokenizer.padding_side = 'right'
    n = len(data['Question'])
    l = []
    for i in range(n):
        conv = []
        conv.append({
            'role': 'user', 'content': data['Question'][i] + ' /think',
        })
        conv.append({
            'role': 'assistant', 'content': '<think>\n' + data['Complex_CoT'][i] + '\n</think>\n\n' +  data['Response'][i]
        })
        l.append(conv)
    return tokenizer.apply_chat_template(l, return_tensors='pt', padding=True, truncation=True)
    

In [ ]:
def prep_data_inference(tokenizer, data, enable_thinking=False):
    tokenizer.padding_side='left'
    n = len(data['Question'])
    l = []
    
    for i in range(n):
        conv = []
        if enable_thinking:
            conv.append({
                'role': 'user', 'content': data['Question'][i] + ' /think',
            })
        else:   
            conv.append({
                'role': 'user', 'content': data['Question'][i],
            })
        l.append(conv)
    return tokenizer.apply_chat_template(l, return_tensors='pt', padding=True, truncation=True)

In [8]:
q_tokens = prep_data_inference(tokenizer, ds['train'][:3]).to(device)

In [9]:
tokenizer.decode(q_tokens['input_ids'])

['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|im_start|>user\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain 

In [10]:
res = model.generate(**q_tokens, max_new_tokens=300)

In [11]:
print(tokenizer.decode(res[0], skip_special_tokens=True))

user
Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?


Okay, let's see. The user is experiencing sudden weakness in the left and right arms, recent long-distance travel, and swollen, tender right lower leg. They want to know what specific cardiac abnormality is likely found during further evaluation. 

First, I need to think about possible cardiac causes. Sudden weakness could be related to heart muscle issues, like a myocardial infarction (heart attack). But wait, long-distance travel is a risk factor for myocardial infarction. The right leg swelling and tenderness could be related to a blood clot or deep vein thrombosis (DVT), which is more common in the lower extremities. 

But wait, DVT is more of a venous problem. However, the patient is experiencing swelling in th

In [12]:
for i in range(3):
    print("------------------------------------------------------------------")
    print('Question:', ds['train'][i]['Question'])
    print('Correct Answer:', ds['train'][i]['Response'])
    print('LLM Answer:', tokenizer.decode(res[i,q_tokens['input_ids'].shape[1]:]))
    print('Think:', ds['train'][i]['Complex_CoT'])
    

------------------------------------------------------------------
Question: Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?
Correct Answer: The specific cardiac abnormality most likely to be found in this scenario is a patent foramen ovale (PFO). This condition could allow a blood clot from the venous system, such as one from a deep vein thrombosis in the leg, to bypass the lungs and pass directly into the arterial circulation. This can occur when the clot moves from the right atrium to the left atrium through the PFO. Once in the arterial system, the clot can travel to the brain, potentially causing an embolic stroke, which would explain the sudden weakness in the left arm and leg. The connection between the recent travel, which increases the risk of deep vein thrombo

In [ ]:
from torch import nn
batch_size = 4
n_epochs = 2
n_train = 13760
n_batches = n_train // batch_size
loss = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=3e-5)
losses = []

for i in range(n_epochs):
    print(f"Start {i} epoch----------------------------------------------------")
    ds_split['train'].shuffle()
    for j in range(n_batches):
        start_idx = j * batch_size
        end_idx = (j+1) * batch_size
        input_tokens = prep_data_train(tokenizer, ds_split['train'][start_idx:end_idx]).to(device)
        print(input_tokens['input_ids'].shape)
        mask = torch.cumsum(input_tokens['input_ids'] == 151644, dim=-1)>=2
        
        logits = model(**input_tokens).logits[:,:-1]
        targets = input_tokens['input_ids'][:,1:]
        
        opt.zero_grad()
        cur_loss = loss(logits[mask[:, 1:]], targets[mask[:, 1:]])
        losses.append(cur_loss.item())
        cur_loss.backward()
        opt.step()
        if j%10 == 0:
            print(f"Loss = {np.array(losses).mean()}")
            losses = []
                
        

Start 0 epoch----------------------------------------------------
torch.Size([4, 704])
Loss = 3.390625
torch.Size([4, 1094])
torch.Size([4, 660])
torch.Size([4, 906])


In [16]:
input_tokens = prep_data_inference(tokenizer, ds_split['train'][:5])

In [17]:
res_new = model.generate(**input_tokens.to(device), max_new_tokens=300)

In [18]:
RED = '\033[31m'
GREEN = '\033[32m'
RESET = '\033[0m'

In [19]:
for i in range(5):
    print(f'{RED}Question: {RESET}', ds_split['train'][i]['Question'])
    print(f'{RED}Correct Answer {RESET}: ', ds_split['train'][i]['Response'])
    print(f'{RED}CoT: {RESET}', ds_split['train'][i]['Complex_CoT'])
    print(f'{GREEN}LLM Answer:', tokenizer.decode(res_new[i, input_tokens['input_ids'].shape[1]:]))

Question:  Considering a 32-year-old woman with a 3-week history of bilateral hand pain and stiffness, primarily in the mornings, swelling and tenderness in the wrists and metacarpophalangeal joints, and subcutaneous nodules on the forearm, what is the most appropriate initial pharmacotherapy to alleviate her current symptoms?
Correct Answer :  Given the clinical presentation, it's important to provide relief from the acute symptoms while also considering long-term management. In this situation, the most appropriate initial pharmacotherapy would be corticosteroids, such as prednisone. Corticosteroids are effective in quickly reducing the inflammation and alleviating the pain and stiffness associated with rheumatoid arthritis, providing the patient with rapid symptom relief. This allows time for disease-modifying antirheumatic drugs (DMARDs) to begin their longer-term action in preventing joint damage.
CoT:  Okay, so we have a 32-year-old woman who's been dealing with some really annoyi

In [41]:
tokenizer.decode(input_tokens['input_ids'])

["<|im_start|>user\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings? /think<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot 

In [45]:
res = model.generate(**q_tokens, max_new_tokens=300)

In [68]:
tokenizer.decode(input_tokens['input_ids'][0][mask[0]])

"<|im_start|>assistant\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to the left side of the heart, it could shoot off to the brain and cause that sudden weakness by blocking blood flow there.\n\nHmm, but how would the clot get from the right side of the heart to the left without going through the lungs and getting filtered out?\n\nHere's wher

In [62]:
q_tokens['input_ids'] = torch.tensor(q_tokens['input_ids']).to(device)
q_tokens['attention_mask'] = torch.tensor(q_tokens['attention_mask']).to(device)

In [63]:
q_tokens['attention_mask']

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1,

In [64]:
cot_tokens['input_ids'] = torch.tensor(cot_tokens['input_ids']).to(device)
cot_tokens['attention_mask'] = torch.tensor(cot_tokens['attention_mask']).to(device)

In [67]:
cot_tokens['attention_mask']

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')

In [76]:
ds['train'][:3]['Question']

['Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',
 'A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?',
 'A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?']

In [78]:
tokenizer.decode(q_tokens['input_ids'])

['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',


In [93]:
tokenizer.decode(q_tokens['input_ids'][:, :-2])

['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these',
 'A 33-yea

In [95]:
[tokenizer.decode(x) for x in (model(**q_tokens).logits[:,-3,:]).argmax(dim=-1)]

[' symptoms', ' injured', 'actions']

In [10]:
from torch import nn

class Qwen3SFT(nn.Module):
    def __init__(self):

    def forward(self, x):
        
    

DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 19704
    })
})